# 02 — MLP Multi-class Classifier (no resampling)

**Strategy.** No GAN. No undersampling. We let the *loss function* — not the
data distribution — handle the class imbalance:

1. **Class weights** (default) — `sklearn`-balanced weights, passed to
   `model.fit(class_weight=...)`. Errors on rare classes get scaled up
   proportionally.
2. **Focal loss** (alternative) — flip `cfg.MLP_CONFIG["imbalance_strategy"]`
   to `"focal"` to use focal loss instead. Focal loss automatically
   down-weights easy (BENIGN) examples and focuses gradient on hard ones.

Both let the model see the **real** distribution while still learning
minorities. We track **macro-F1** (not accuracy) because accuracy is
dominated by BENIGN.


In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
)

from src import config as cfg
from src.models import build_mlp, compute_balanced_class_weights
sns.set_theme(style="whitegrid")
print("TF:", tf.__version__)
print("Imbalance strategy:", cfg.MLP_CONFIG["imbalance_strategy"])


## 1. Load data, scaler, label map

In [ ]:
train = pd.read_csv(cfg.PROCESSED_DIR / "train.csv")
val   = pd.read_csv(cfg.PROCESSED_DIR / "val.csv")

scaler = joblib.load(cfg.SCALER_FILE)
with open(cfg.FEATURE_NAMES_FILE) as f: feature_names = json.load(f)
with open(cfg.LABEL_MAP_FILE)    as f: label_map = {int(k):v for k,v in json.load(f).items()}
inv_map = {v:k for k,v in label_map.items()}
class_names = [label_map[i] for i in range(len(label_map))]

# Some rows in val may carry rare labels we dropped at preprocessing.
train = train[train[cfg.LABEL_COL].isin(inv_map)].reset_index(drop=True)
val   = val[val[cfg.LABEL_COL].isin(inv_map)].reset_index(drop=True)

X_train = scaler.transform(train[feature_names].astype(np.float32).values)
y_train = train[cfg.LABEL_COL].map(inv_map).astype(int).values
X_val   = scaler.transform(val[feature_names].astype(np.float32).values)
y_val   = val[cfg.LABEL_COL].map(inv_map).astype(int).values

print(f"X_train: {X_train.shape}  X_val: {X_val.shape}  classes: {len(label_map)}")


## 2. Class distribution (kept untouched)

In [ ]:
dist = pd.Series(y_train).value_counts().sort_index()
dist.index = [class_names[i] for i in dist.index]
plt.figure(figsize=(12, 4))
sns.barplot(x=dist.index, y=dist.values, color="steelblue")
plt.yscale("log")
plt.xticks(rotation=60, ha="right")
plt.title("Training class distribution (real, log scale)")
plt.tight_layout(); plt.show()
print(dist)


## 3. Compute class weights

`compute_class_weight('balanced', ...)` gives weight ∝ N / (n_classes * count_class).
We cap at `cfg.MLP_CONFIG['max_class_weight']` so the rarest classes don't
explode the gradient.


In [ ]:
class_weights = compute_balanced_class_weights(
    y_train, max_weight=cfg.MLP_CONFIG["max_class_weight"]
)

cw_df = pd.DataFrame([
    {"class": class_names[i], "count": int((y_train == i).sum()), "weight": class_weights[i]}
    for i in sorted(class_weights)
]).sort_values("weight", ascending=False)
print(cw_df.to_string(index=False))


## 4. Build the MLP

In [ ]:
strategy = cfg.MLP_CONFIG["imbalance_strategy"]
loss_kind = "focal" if strategy == "focal" else "sparse_categorical_crossentropy"

model = build_mlp(
    input_dim=X_train.shape[1],
    n_classes=len(label_map),
    hidden_units=cfg.MLP_CONFIG["hidden_units"],
    dropout=cfg.MLP_CONFIG["dropout"],
    learning_rate=cfg.MLP_CONFIG["learning_rate"],
    loss=loss_kind,
    focal_gamma=cfg.MLP_CONFIG["focal_gamma"],
    focal_alpha=cfg.MLP_CONFIG["focal_alpha"],
)
model.summary()


## 5. Macro-F1 callback (the metric that actually matters)

In [ ]:
class MacroF1Callback(tf.keras.callbacks.Callback):
    """Compute macro-F1 on the validation set after each epoch."""
    def __init__(self, X, y):
        super().__init__()
        self.X, self.y = X, y
        self.history = []
    def on_epoch_end(self, epoch, logs=None):
        y_pred = self.model.predict(self.X, batch_size=4096, verbose=0).argmax(axis=1)
        score = f1_score(self.y, y_pred, average="macro", zero_division=0)
        logs = logs or {}
        logs["val_macro_f1"] = score
        self.history.append(score)
        print(f"  -> val_macro_f1: {score:.4f}")

f1_cb = MacroF1Callback(X_val, y_val)


## 6. Train

In [ ]:
cfg.MODELS_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    f1_cb,
    EarlyStopping(monitor="val_macro_f1", mode="max",
                  patience=cfg.MLP_CONFIG["patience"], restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ModelCheckpoint(filepath=str(cfg.MLP_MODEL_FILE),
                    monitor="val_macro_f1", mode="max", save_best_only=True),
]

# class_weight is only used for the cross-entropy strategy.
fit_kwargs = {}
if strategy == "class_weights":
    fit_kwargs["class_weight"] = class_weights
    print("Training with sklearn-balanced class weights.")
elif strategy == "focal":
    print(f"Training with focal loss (gamma={cfg.MLP_CONFIG['focal_gamma']}, "
          f"alpha={cfg.MLP_CONFIG['focal_alpha']}).")
else:
    print("WARNING: imbalance_strategy='none' — model will likely collapse to BENIGN.")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=cfg.MLP_CONFIG["epochs"],
    batch_size=cfg.MLP_CONFIG["batch_size"],
    callbacks=callbacks,
    verbose=2,
    **fit_kwargs,
)


## 7. Learning curves

In [ ]:
h = history.history
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(h["loss"], label="train"); axes[0].plot(h["val_loss"], label="val")
axes[0].set_title("loss"); axes[0].legend()
axes[1].plot(h["accuracy"], label="train"); axes[1].plot(h["val_accuracy"], label="val")
axes[1].set_title("accuracy"); axes[1].legend()
axes[2].plot(f1_cb.history, color="seagreen")
axes[2].set_title("val macro-F1"); axes[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()


## 8. Validation report (per-class & macro)

In [ ]:
y_pred = model.predict(X_val, batch_size=4096, verbose=0).argmax(axis=1)

print(f"Macro F1   : {f1_score(y_val, y_pred, average='macro',    zero_division=0):.4f}")
print(f"Weighted F1: {f1_score(y_val, y_pred, average='weighted', zero_division=0):.4f}\n")
print(classification_report(y_val, y_pred, target_names=class_names,
                            digits=4, zero_division=0))


## 9. Confusion matrix

In [ ]:
cm = confusion_matrix(y_val, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

plt.figure(figsize=(11, 9))
sns.heatmap(cm_norm, xticklabels=class_names, yticklabels=class_names,
            annot=True, fmt=".2f", cmap="Blues", cbar=False)
plt.xlabel("predicted"); plt.ylabel("true")
plt.title("Validation — normalised confusion matrix")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()


## 10. Persist the best model

In [ ]:
# ModelCheckpoint already wrote the best epoch; re-save in canonical location
# (in case the in-memory `model` differs from the checkpoint).
model.save(cfg.MLP_MODEL_FILE)
print("Saved MLP ->", cfg.MLP_MODEL_FILE)


---
**Next:** `03_autoencoder_training.ipynb` for the unsupervised anomaly net.
